In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
from T_method import LayeredStructure
from My_plotter import Plotter, Style
from Global_optimizer import my_json_load 

In [ ]:
MU_0 = 4e-7 * np.pi
EPS_0 = 8.8541878188e-12
ETA_0 = np.sqrt(MU_0/EPS_0)
C = 1/np.sqrt(MU_0*EPS_0)

In [ ]:
a = 8e-3
f_min = 3.3e9
f_max = 4.2e9
f_0 = (f_min + f_max)/2
print(f'центральная частота - {(f_0*1.e-9):.2f} ГГц')
lamb_0 = C/f_0
k_0 = 2*np.pi/lamb_0
print(f'длина волны на центральной частоте - {(lamb_0*1000):.2f} мм')
print(f'волновое число на центральной частоте - {(k_0):.2f} 1/м')
delta = a*3

In [ ]:
st = Style()
fig, ax = plt.subplots()
pl = Plotter(ax, st)
df = np.linspace(-50, 50, 400)/100
alpha = np.array([1.5, 1.5, 1.5])
structure = LayeredStructure(alpha, beta_d=np.pi)
directivity = 10*np.log10(structure.directivity(df))
directivity_two_sources = 10*np.log10(structure.directivity_two_sources_diagonal(df))
pl.plot(df, directivity, label='Directivity')
pl.plot(df, directivity_two_sources, label='Directivity (Two Sources)', linestyle='--')
pl.finalize()
ax.axhline(19, color='gray', linestyle='--', alpha=0.5)
plt.show()

In [ ]:
theta = np.array([1,2])
df = np.array([0, 0.1, 0.2])
res = theta[None, :, None] + df[None, None, :]
print(np.shape(res))
print(np.shape(res[:, None]))

In [ ]:
import scipy as sc
from scipy.special import j0, j1
y = np.linspace(0, 1, 200)
def f(y, real=True):
    if real:
        return lambda x: np.cos(x)**2*np.cos(y*(np.sin(x)+np.cos(x)))
    return lambda x: np.cos(x)**2*np.sin(y*(np.sin(x)+np.cos(x)))

resf = np.zeros_like(y, dtype=np.complex128)

for yi in range(len(y)):
    resf[yi] = sc.integrate.quad(f(y[yi], real=True), 0, np.pi*2)[0] + 1.j*sc.integrate.quad(f(y[yi], real=False), 0, np.pi*2)[0]

def integral_result(y):
    # Обработка y = 0 (предел)
    if np.isscalar(y) and y == 0:
        return np.pi / 2  # предел при y->0
    # Общий случай
    sqrt2_y = np.sqrt(2) * y
    return np.pi * (j0(sqrt2_y))

res_analytical = np.array([integral_result(yi) for yi in y])

plt.plot(y, resf, label='Numerical Integration')
plt.plot(y, res_analytical, label='Analytical Result', linestyle='--')
plt.xlabel('y')
plt.ylabel('Integral Result')
plt.legend()
plt.title('Comparison of Numerical and Analytical Integration Results')
plt.show()


In [ ]:
from scipy.special import jv
# Evaluate the 0th-order Bessel function at x=1
print(jv(0, 0)) 

In [ ]:
st = Style()
fig, ax = plt.subplots()
pl1 = Plotter(ax, st)
df = np.linspace(-30, 30, 200)/100
for seed in [90]:
    path = Path(r"results\final\cc"+f"{seed}.json")
    data = my_json_load(path)
    alpha = data["best_alpha"]
    beta = data["best_beta"]
    alpha_l = data["best_alpha_l"]
    alpha_c = data["best_alpha_c"]
    dipole_shift = data["best_dipole_shift"]
    beta_d = data["best_beta_d"]

    structure = LayeredStructure(alpha, beta=beta, alpha_l=alpha_l, alpha_c=alpha_c, dipole_shift=dipole_shift, beta_d=beta_d)
    directivity = 10*np.log10(structure.directivity_two_sources_diagonal(df))
    pl1.plot((1+df)*f_0*1.e-9, directivity, label=f"18dBi")
pl1.finalize()
pl1.set_ylim((0, 25))
ax.axhline(18, color='gray', linestyle='--', alpha=0.5)
ax.axhline(16, color='gray', linestyle='--', alpha=0.5)
ax.axvline(f_max*1.e-9, color='gray', linestyle='--', alpha=0.5)
ax.axvline(f_min*1.e-9, color='gray', linestyle='--', alpha=0.5)
# ax.axvline((1-0.15)*f_0*1.e-9, color='orange', linestyle='--', alpha=0.5)
# ax.axvline((1+0.15)*f_0*1.e-9, color='orange', linestyle='--', alpha=0.5)
ax.axvline((1)*f_0*1.e-9, color='orange', linestyle='--', alpha=0.5)
ax.axvline
ax.set_xlabel('Frequency (GHz)')
ax.set_ylabel('Directivity (dBi)')
ax.minorticks_on()
plt.show()

In [ ]:
print(f'best_alpha_Z = {ETA_0/data["best_alpha"]}')
print(f'best_beta_l = {data["best_beta"]/k_0*1000}')
print(f'best_alpha_l_Z = {ETA_0/data["best_alpha_l"]}')
print(f'best_alpha_c_Z = {ETA_0/data["best_alpha_c"]}')
print(f'best_dipole_shift_l = {data["best_dipole_shift"]/k_0*1000}')
print(f'best_beta_d_l = {data["best_beta_d"]/k_0*1000}')


In [ ]:
import csv
f_cst = np.linspace(3.0, 4.5, 200)
x_1 = ETA_0/data["best_alpha"]*(f_cst/f_0*1.e9)
weights = np.ones_like(f_cst)
r = np.zeros_like(f_cst)
data_z1 = np.array([f_cst, r, x_1, weights]).T
path = Path(r"dispersion_for_cst\z_1.txt")
path.parent.mkdir(parents=True, exist_ok=True)
with open(path, 'w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file, delimiter='\t')
    writer.writerows(data_z1.tolist())

In [ ]:
f_cst = np.linspace(3.0, 4.5, 200)
y_l = data["best_alpha_l"]/ETA_0
y_c = data["best_alpha_c"]/ETA_0
f_rel = f_cst/f_0*1.e9
x_scr = 1/(y_l/f_rel+ y_c*f_rel)
weights = np.ones_like(f_cst)
r = np.zeros_like(f_cst)
data_z1 = np.array([f_cst, r, x_scr, weights]).T
path = Path(r"dispersion_for_cst\z_scr.txt")
path.parent.mkdir(parents=True, exist_ok=True)
with open(path, 'w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file, delimiter='\t')
    writer.writerows(data_z1.tolist())

In [ ]:
def radiation_pattern(self, phi, theta, df, mode='normalized'):
    def p_on_direction_array(phi, theta, df):
        T_shift = FreeSpaceLayer(np.pi/2, theta, df).Tmatrix() # [phi][theta][df][*][*]
        vec_0 = np.array([0,1])[None, None, None, :,None]
        p_s = ((T_shift@vec_0).transpose(4,3,2,0,1)[0, 0, :, :, :])**2 # [df][phi][theta]
        vec_TM = np.array([0,1])[None, None, None, :,None]
        vec_TE = np.array([0,1])[None, None, None, :,None]
        for i in range(self.N):
            vec_TM = FreeSpaceLayer(self.beta[i], theta, df).Tmatrix()@vec_TM
            vec_TM = ImpSheetLayer(self.alpha[i], theta, df, 'TM', self.dispersion[i]).Tmatrix()@vec_TM
            vec_TE = FreeSpaceLayer(self.beta[i], theta, df).Tmatrix()@vec_TE
            vec_TE = ImpSheetLayer(self.alpha[i], theta, df, 'TE', self.dispersion[i]).Tmatrix()@vec_TE
        vec_TM = vec_TM.transpose(4, 3, 2, 0, 1)[0, :, :, :, :] 
        vec_TE = vec_TE.transpose(4, 3, 2, 0, 1)[0, :, :, :, :]
        # удалена фиктивная матричная ось, теперь [компоненты вектора][df][phi][theta]
        p_TM = (vec_TM[0, :, :, :]**2 + vec_TM[1, :, :, :]**2) # [df][phi][theta]
        p_TE = (vec_TE[0, :, :, :]**2 + vec_TE[1, :, :, :]**2)
        e_xi_TM = np.sqrt(p_s/p_TM)
        e_xi_TE = np.sqrt(p_s/p_TE)
        phi = phi[None, :, None]
        #диполь ориентирован вдоль оси x
        p = (np.cos(theta)**2*np.cos(phi)**2*e_xi_TM**2 + np.sin(phi)**2*e_xi_TE**2)
        return p
    p_total = p_on_direction_array(phi, theta, df)/p_on_direction_array(np.array([0]), np.array([0]), df)
    if mode == 'normalized':
        return p_total
    elif mode == 'absolute':
        return p_total*(self.directivity(df)[:, None, None])